# UC12 — Identificador de Oportunidades de Subrogación

Modelo ML para identificar siniestros con alta probabilidad de recuperación por subrogación.
**Valor de negocio**: Recuperar millones en pérdidas pagadas identificando terceros responsables.

---

## Paso 1: Configuración del Entorno

In [ ]:
CREATE DATABASE IF NOT EXISTS SUBROGACION_DB;
USE SCHEMA SUBROGACION_DB.PUBLIC;
CREATE WAREHOUSE IF NOT EXISTS COMPUTE_WH WITH WAREHOUSE_SIZE='SMALL' AUTO_SUSPEND=60 AUTO_RESUME=TRUE;
USE WAREHOUSE COMPUTE_WH;

---

## Paso 2: Generar Siniestros Cerrados (1.200)

In [ ]:
CREATE OR REPLACE TABLE siniestros_cerrados AS
SELECT 'SIN' || LPAD(SEQ4(),6,'0') AS siniestro_id,
    CASE MOD(SEQ4(),4) WHEN 0 THEN 'Colision' WHEN 1 THEN 'Alcance' WHEN 2 THEN 'Salida_Via' ELSE 'Aparcamiento' END AS tipo_incidente,
    CASE WHEN UNIFORM(0,1,RANDOM())<0.6 THEN 1 ELSE 0 END AS tiene_atestado,
    CASE MOD(UNIFORM(1,100,RANDOM()),4) WHEN 0 THEN 'Total' WHEN 1 THEN 'Parcial' WHEN 2 THEN 'Ninguna' ELSE 'Desconocida' END AS culpabilidad_tercero,
    CASE WHEN UNIFORM(0,1,RANDOM())<0.5 THEN 1 ELSE 0 END AS hay_testigos,
    UNIFORM(1,4,RANDOM()) AS num_vehiculos,
    CASE MOD(UNIFORM(1,100,RANDOM()),3) WHEN 0 THEN 'Todo_Riesgo' WHEN 1 THEN 'Terceros_Ampliado' ELSE 'Terceros' END AS tipo_cobertura,
    UNIFORM(200,30000,RANDOM())::DECIMAL(10,2) AS importe_pagado,
    CASE WHEN UNIFORM(0,1,RANDOM())<0.7 THEN 1 ELSE 0 END AS tiene_informe_perito,
    UNIFORM(5,180,RANDOM()) AS dias_tramitacion,
    CASE WHEN UNIFORM(0,1,RANDOM())<0.20 THEN 1 ELSE 0 END AS se_recupero
FROM TABLE(GENERATOR(ROWCOUNT=>1200));

---

## Paso 3: Feature Engineering

In [ ]:
CREATE OR REPLACE TABLE features_subrogacion AS
SELECT siniestro_id, tiene_atestado::FLOAT AS tiene_atestado,
    CASE WHEN culpabilidad_tercero='Total' THEN 1.0 ELSE 0.0 END AS culpabilidad_total,
    CASE WHEN culpabilidad_tercero='Parcial' THEN 1.0 ELSE 0.0 END AS culpabilidad_parcial,
    hay_testigos::FLOAT AS hay_testigos, num_vehiculos::FLOAT AS num_vehiculos,
    CASE WHEN tipo_cobertura='Todo_Riesgo' THEN 1.0 ELSE 0.0 END AS es_todo_riesgo,
    importe_pagado::FLOAT AS importe_pagado, tiene_informe_perito::FLOAT AS tiene_informe,
    dias_tramitacion::FLOAT AS dias_tramitacion,
    CASE WHEN tipo_incidente='Alcance' THEN 1.0 ELSE 0.0 END AS es_alcance,
    CASE WHEN tipo_incidente='Colision' THEN 1.0 ELSE 0.0 END AS es_colision,
    se_recupero
FROM siniestros_cerrados;

---

## Paso 4: Train/Test

In [ ]:
CREATE OR REPLACE TABLE entrenamiento AS SELECT * FROM features_subrogacion SAMPLE(80);
CREATE OR REPLACE TABLE test AS SELECT * FROM features_subrogacion WHERE siniestro_id NOT IN (SELECT siniestro_id FROM entrenamiento);

---

## Paso 5: Entrenar Modelo

In [ ]:
CREATE OR REPLACE SNOWFLAKE.ML.CLASSIFICATION detector_subrogacion(
    INPUT_DATA=>SYSTEM$REFERENCE('TABLE','entrenamiento'),
    TARGET_COLNAME=>'se_recupero',
    CONFIG_OBJECT=>{'evaluate':TRUE}
);

---

## Paso 6: Evaluar

In [ ]:
CALL detector_subrogacion!SHOW_EVALUATION_METRICS();

In [ ]:
CALL detector_subrogacion!SHOW_FEATURE_IMPORTANCE();

---

## Paso 7: Puntuar y Estimar Recuperación

In [ ]:
CREATE OR REPLACE TABLE predicciones_subrogacion AS
SELECT siniestro_id,
    detector_subrogacion!PREDICT(OBJECT_CONSTRUCT(
        'tiene_atestado',tiene_atestado,'culpabilidad_total',culpabilidad_total,'culpabilidad_parcial',culpabilidad_parcial,
        'hay_testigos',hay_testigos,'num_vehiculos',num_vehiculos,'es_todo_riesgo',es_todo_riesgo,
        'importe_pagado',importe_pagado,'tiene_informe',tiene_informe,'dias_tramitacion',dias_tramitacion,
        'es_alcance',es_alcance,'es_colision',es_colision
    )) AS prediccion,
    prediccion['probability']['1']::FLOAT AS prob_recuperacion,
    CASE WHEN prediccion['probability']['1']::FLOAT>=0.65 THEN 'Alta Probabilidad'
         WHEN prediccion['probability']['1']::FLOAT>=0.35 THEN 'Revisable'
         ELSE 'Baja Probabilidad' END AS nivel,
    importe_pagado,
    (importe_pagado*prediccion['probability']['1']::FLOAT)::DECIMAL(10,2) AS importe_recuperable,
    se_recupero AS recuperado_real
FROM test;

SELECT nivel, COUNT(*) AS siniestros, SUM(importe_recuperable)::DECIMAL(12,2) AS total_recuperable FROM predicciones_subrogacion GROUP BY nivel;

---

## Paso 8: Análisis de Oportunidad Perdida

In [ ]:
SELECT nivel, COUNT(*) AS oportunidades, SUM(importe_pagado)::DECIMAL(12,2) AS total_pagado, SUM(importe_recuperable)::DECIMAL(12,2) AS total_recuperable
FROM predicciones_subrogacion WHERE recuperado_real=0 AND nivel='Alta Probabilidad' GROUP BY nivel;

---

## Paso 9: Dashboard Interactivo

In [ ]:
import streamlit as st
import pandas as pd
from snowflake.snowpark.context import get_active_session

session = get_active_session()
st.title('Oportunidades de Subrogación')
df = session.table('predicciones_subrogacion').to_pandas()

c1,c2,c3,c4 = st.columns(4)
with c1: st.metric('Siniestros', len(df))
with c2: st.metric('Alta Probabilidad', len(df[df['NIVEL']=='Alta Probabilidad']))
with c3: st.metric('Recuperable Estimado', f"{df[df['NIVEL']=='Alta Probabilidad']['IMPORTE_RECUPERABLE'].sum():,.0f}€")
with c4: st.metric('Tasa Éxito Histórica', f"{df['RECUPERADO_REAL'].mean():.1%}")

st.markdown('---')
st.subheader('Distribución por Nivel')
rc = df['NIVEL'].value_counts()
st.bar_chart(pd.DataFrame({'Siniestros': rc.values}, index=rc.index))

st.markdown('---')
st.subheader('Top Oportunidades')
top = df[df['NIVEL']=='Alta Probabilidad'].nlargest(20,'IMPORTE_RECUPERABLE')
top['Prob %'] = top['PROB_RECUPERACION'].apply(lambda x: f'{x:.1%}')
top['Recuperable'] = top['IMPORTE_RECUPERABLE'].apply(lambda x: f'{x:,.0f}€')
st.dataframe(top[['SINIESTRO_ID','Prob %','IMPORTE_PAGADO','Recuperable','RECUPERADO_REAL']], use_container_width=True)

st.markdown('---')
csv = df[df['NIVEL']=='Alta Probabilidad'].to_csv(index=False)
st.download_button('Descargar Oportunidades (CSV)', data=csv, file_name='oportunidades_subrogacion.csv', mime='text/csv')
st.caption('Desarrollado con Snowflake Cortex ML + Streamlit')

---

## Paso 10: Limpieza (Opcional)

In [ ]:
-- DROP DATABASE IF EXISTS SUBROGACION_DB;
-- DROP WAREHOUSE IF EXISTS COMPUTE_WH;